# README

- **Author**: 許以琳
- **Created At**: 2026-11-10
- **Last Modified At**: 2026-11-10

---

## What does this file do?

- 以 `candidates_scope_g1_g6_20261108.csv`（267 人分析樣本）為底，套用 G6 政治家族的更新：比對組員張睿庠 `national_g5_g6_all_20261108.csv`（origin/main 43fa665）後，有 31 人本版記 0、組員查到有來源的公職親屬記 1；組員資料有親屬關係、公職、全文來源網址與保存檔，較本版（維基百科無條目或人工記「無」）完整，依許以琳 2026-11-10 決定改採組員的 G6=1。
- 不採用的不一致：劉品妡本版記 1（維基百科載祖父為七屆宜蘭縣議員），組員記 0（搜尋未發現），本版證據較完整，維持 1；曹爾元組員為未完成（未知），維持本版 0。
- 只更新 G6 相關欄位（`G6_political_family`、`G6_label`、`G6_evidence`、`G6_source`、`G6_evidence_tier`），並新增 `G6_updated_20261110`（1＝本次改採組員資料）與 `G6_review_flag_20261110`（需再討論的定義邊界）。其他欄位、樣本、列數均不變；原 20261108 檔不覆寫。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/processed/candidates_scope_g1_g6_20261108.csv`
     - Input key: `scope`
     - Description: 267 位候選人的 G1–G6 分析檔，鍵 `candidate_key`。
     - Documentation: `data/code/merge_scope_g1_g6_20261108.ipynb`
     - SHA-256: `5af31805a707502198004aad222ac9a61c4309d9bc27cfa1c059450d74b8e6a6`
  2. `data/raw/student_decisions_20261108/g6_teammate_overrides_20261110.csv`
     - Input key: `overrides`
     - Description: 31 人的 G6 改採紀錄：原值、新值、組員的親屬與公職證據、來源網址與保存檔、證據等級、需再討論的旗標，鍵 `candidate_key`。由組員 `data/processed/national_g5_g6_all_20261108.csv`（origin/main 43fa665）逐列抽出。
     - Documentation: 本 notebook
     - SHA-256: `808c69647537d1a2a677a16686fa69a3795fac34818c9b84b965723d56ad73c6`

---

## What does this file output?

- `data/processed/candidates_scope_g1_g6_20261110.csv`
  - Output key: `scope_v2`
  - Description: 267 人 G1–G6 分析檔，G6 已依組員證據更新 31 人。
  - Role: `analysis`
  - Documentation: 本 notebook（processed-data documentation 待建立）
  - Expected schema and row-count rules: 267 列、`candidate_key` 唯一；欄位為原檔全部欄位＋`G6_updated_20261110`、`G6_review_flag_20261110`；只有 31 人的 G6 由 0 改為 1，其他值不變。
  - Verification status / SHA-256: 見下方執行結果
  - Downstream finding input: `data/processed/candidates_scope_g1_g6_20261110.csv`

All paths are relative to the project root, not the operating-system root.


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


In [2]:
INPUTS = {
    "scope": "data/processed/candidates_scope_g1_g6_20261108.csv",
    "overrides": "data/raw/student_decisions_20261108/g6_teammate_overrides_20261110.csv",
}
EXPECTED_INPUT_SHA256 = {
    "scope": "5af31805a707502198004aad222ac9a61c4309d9bc27cfa1c059450d74b8e6a6",
    "overrides": "808c69647537d1a2a677a16686fa69a3795fac34818c9b84b965723d56ad73c6",
}
OUTPUTS = {"scope_v2": "data/processed/candidates_scope_g1_g6_20261110.csv"}
INPUT_OPTIONS = {key: {"dtype": str, "encoding": "utf-8-sig"} for key in INPUTS}
OUTPUT_READ_OPTIONS = {key: {"dtype": str, "encoding": "utf-8-sig"} for key in OUTPUTS}

input_paths = {}
for key, relative_path in INPUTS.items():
    area = Path(relative_path).parts[1] if len(Path(relative_path).parts) > 1 else ""
    if area not in {"raw", "processed"}:
        raise ValueError(f"Expected a raw or upstream processed input: {relative_path}")
    input_paths[key] = project_file(relative_path, area)
output_paths = {key: project_file(path, "processed", must_exist=False) for key, path in OUTPUTS.items()}
if len(set(output_paths.values())) != len(output_paths):
    raise ValueError("Each output needs a distinct file path.")
if set(input_paths.values()) & set(output_paths.values()):
    raise ValueError("An output must not overwrite one of its inputs.")
existing = [str(path) for path in output_paths.values() if path.exists()]
if existing:
    raise FileExistsError(f"Review existing outputs before replacing them: {existing}")

if set(EXPECTED_INPUT_SHA256) != set(INPUTS):
    raise ValueError("Every input must have its own expected SHA-256.")
input_fingerprints = {key: verify_input_sha256(path, EXPECTED_INPUT_SHA256[key])
                      for key, path in input_paths.items()}
inputs = {key: pd.read_csv(path, **INPUT_OPTIONS.get(key, {})) for key, path in input_paths.items()}
for key, frame in inputs.items():
    print({"input": key, "data_path": INPUTS[key], "rows": len(frame),
           "schema": {column: str(dtype) for column, dtype in frame.dtypes.items()},
           "sha256": input_fingerprints[key]})


{'input': 'scope', 'data_path': 'data/processed/candidates_scope_g1_g6_20261108.csv', 'rows': 267, 'schema': {'candidate_key': 'str', 'candidate_name': 'str', 'county_city': 'str', 'office_type': 'str', 'electoral_district': 'str', 'G1_age_2026_minus_birth_year': 'str', 'birth_date': 'str', 'birth_year': 'str', 'G1_source_tier': 'str', 'G1_source': 'str', 'G2_sex': 'str', 'G2_source_tier': 'str', 'G2_source': 'str', 'G3_education_level': 'str', 'education_raw': 'str', 'G3_source': 'str', 'G3_note': 'str', 'education_source_tier': 'str', 'education_historical_level': 'str', 'G4_party': 'str', 'party_registration_raw': 'str', 'G4_source': 'str', 'G4_note': 'str', 'G5_first_time_any_elected_office': 'str', 'G5_label': 'str', 'G5_prior_election_year_example': 'str', 'G5_evidence': 'str', 'G5_source': 'str', 'G5_audit_status': 'str', 'G6_political_family': 'str', 'G6_label': 'str', 'G6_evidence': 'str', 'G6_source': 'str', 'G6_evidence_tier': 'str', 'source_conflict_flag': 'str', 'checked_d

In [3]:
scope = inputs["scope"].copy()
ov = inputs["overrides"]
assert scope["candidate_key"].is_unique and ov["candidate_key"].is_unique
assert set(ov["candidate_key"]) <= set(scope["candidate_key"])
assert (ov["G6_old"] == "0").all() and (ov["G6_new"] == "1").all()
before = scope.copy()

scope["G6_updated_20261110"] = "0"
scope["G6_review_flag_20261110"] = pd.Series([None] * len(scope), dtype=object)
idx = scope.set_index("candidate_key").index
for r in ov.itertuples(index=False):
    i = idx.get_loc(r.candidate_key)
    assert scope.at[i, "G6_political_family"] == r.G6_old, f"{r.candidate_name} 原值與紀錄不符"
    scope.at[i, "G6_political_family"] = r.G6_new
    scope.at[i, "G6_label"] = "有三親等內或配偶公職來源記載"
    scope.at[i, "G6_evidence"] = r.teammate_G6_evidence
    scope.at[i, "G6_source"] = r.teammate_G6_source
    scope.at[i, "G6_evidence_tier"] = "teammate_" + r.teammate_G6_evidence_tier
    scope.at[i, "G6_updated_20261110"] = "1"
    scope.at[i, "G6_review_flag_20261110"] = r.review_flag

changed_cols = ["G6_political_family", "G6_label", "G6_evidence", "G6_source", "G6_evidence_tier"]
other = [c for c in before.columns if c not in changed_cols]
pd.testing.assert_frame_equal(before[other], scope[other])                       # 其他欄位不變
diff = (before["G6_political_family"] != scope["G6_political_family"])
assert diff.sum() == len(ov) == 31
assert set(scope.loc[diff, "candidate_key"]) == set(ov["candidate_key"])
print("G6 更新人數：", int(diff.sum()))
print("G6 分布 前：", before["G6_political_family"].value_counts().to_dict(), "→ 後：", scope["G6_political_family"].value_counts().to_dict())
print(pd.crosstab(scope["county_city"].isin(["臺北市", "高雄市"]).map({True: "臺北＋高雄", False: "其他縣市"}),
                  scope["G6_political_family"], margins=True, margins_name="合計"))
print("需再討論的定義邊界：")
print(scope.loc[scope["G6_review_flag_20261110"].notna(), ["candidate_name", "G6_review_flag_20261110"]].to_string())

scope = scope.astype("str")
outputs = {"scope_v2": scope}


G6 更新人數： 31
G6 分布 前： {'0': 211, '1': 56} → 後： {'0': 180, '1': 87}
G6_political_family    0   1   合計
county_city                      
其他縣市                  46  23   69
臺北＋高雄                134  64  198
合計                   180  87  267
需再討論的定義邊界：
    candidate_name                    G6_review_flag_20261110
1              沈伯洋  親屬為曾祖父（三親等直系），公職為日治時期村里長，屬歷史地方公職，納入與否可再討論
51              陳瑩                           父親職位為政務任命（原民會主委）
120             許甫                       配偶職位為政府任命（臺北市政府副發言人）
244            吳銘賜                     姊妹關係由同篇報導的共同姪輩推定，非直接明載


In [4]:
# Run only after the transformation cell has produced and checked `outputs`.
assert set(outputs) == set(OUTPUTS), "Produced datasets must match the declared output paths."
for key, frame in outputs.items():
    if not isinstance(frame, pd.DataFrame) or not frame.columns.is_unique:
        raise ValueError(f"Output {key} must be a DataFrame with unique column names.")

verification = {}
for key, frame in outputs.items():
    path = output_paths[key]
    path.parent.mkdir(parents=True, exist_ok=True)
    # Exclusive creation prevents an accidental overwrite during a rerun.
    with path.open("x", encoding="utf-8-sig", newline="") as stream:
        frame.to_csv(stream, index=False)
    reread = pd.read_csv(path, **OUTPUT_READ_OPTIONS.get(key, {}))
    assert list(reread.columns) == list(frame.columns), f"Schema changed while saving {key}."
    assert len(reread) == len(frame), f"Row count changed while saving {key}."
    # Adapt storage/type normalization explicitly when needed; preserve values.
    pd.testing.assert_frame_equal(frame.reset_index(drop=True), reread,
                                  check_dtype=False, check_categorical=False)
    verification[key] = {"data_path": OUTPUTS[key], "rows": len(reread),
                         "schema": {column: str(dtype) for column, dtype in reread.dtypes.items()},
                         "sha256": sha256_file(path), "status": "read-back checks passed"}
    print(verification[key])
assert all(sha256_file(path) == input_fingerprints[key] for key, path in input_paths.items()), "An input changed."
# Use these actual results to finish every declared processed-data document
# and Data Artifacts row. A failed write/read-back remains unverified.
# Finding notebooks must use the exact output path, matching role and SHA-256,
# never the first CSV, a raw source, or a fabricated example path.


{'data_path': 'data/processed/candidates_scope_g1_g6_20261110.csv', 'rows': 267, 'schema': {'candidate_key': 'str', 'candidate_name': 'str', 'county_city': 'str', 'office_type': 'str', 'electoral_district': 'str', 'G1_age_2026_minus_birth_year': 'str', 'birth_date': 'str', 'birth_year': 'str', 'G1_source_tier': 'str', 'G1_source': 'str', 'G2_sex': 'str', 'G2_source_tier': 'str', 'G2_source': 'str', 'G3_education_level': 'str', 'education_raw': 'str', 'G3_source': 'str', 'G3_note': 'str', 'education_source_tier': 'str', 'education_historical_level': 'str', 'G4_party': 'str', 'party_registration_raw': 'str', 'G4_source': 'str', 'G4_note': 'str', 'G5_first_time_any_elected_office': 'str', 'G5_label': 'str', 'G5_prior_election_year_example': 'str', 'G5_evidence': 'str', 'G5_source': 'str', 'G5_audit_status': 'str', 'G6_political_family': 'str', 'G6_label': 'str', 'G6_evidence': 'str', 'G6_source': 'str', 'G6_evidence_tier': 'str', 'source_conflict_flag': 'str', 'checked_date': 'str', 'G1_f